# GitHub Sentiment Monitor — запуск в Google Colab

Загрузите ZIP-архив проекта, установите зависимости и запустите Streamlit через встроенный прокси Google Colab. В приложении можно перечислять репозитории через запятую или с новой строки, например `pallets/flask,streamlit/streamlit,psf/requests`.


In [ ]:
from google.colab import files
uploaded = files.upload()  # Выберите github_sentiment_streamlit.zip
archive_name = next(name for name in uploaded if name.lower().endswith('.zip'))
print('Загружен архив:', archive_name)

In [ ]:
import zipfile
from pathlib import Path
with zipfile.ZipFile(archive_name) as archive:
    archive.extractall('/content')
project_dir = Path('/content/github_sentiment_streamlit')
assert (project_dir / 'app.py').exists(), 'В архиве не найдена папка проекта'
%cd /content/github_sentiment_streamlit

In [ ]:
!pip -q install -r requirements.txt

## Запуск Streamlit

Следующая ячейка проверяет расположение проекта, запускает Streamlit и открывает интерфейс через встроенный прокси Google Colab. LocalTunnel и страница с кнопкой Continue не используются. Не публикуйте GitHub token в ноутбуке или открытых ячейках — вводите его непосредственно в поле приложения.

In [ ]:
import os
import subprocess
import time
from pathlib import Path
import requests
from google.colab import output

# После перезапуска среды Colab рабочая директория может сброситься.
project_dir = Path("/content/github_sentiment_streamlit")
if not (project_dir / "app.py").is_file():
    raise FileNotFoundError(
        "Не найдена папка проекта /content/github_sentiment_streamlit. "
        "Повторно выполните ячейки загрузки и распаковки ZIP-архива."
    )
os.chdir(project_dir)
print("Рабочая директория:", Path.cwd())

# Если Streamlit уже запущен (например, ячейку выполняют повторно), используем его.
health_url = "http://127.0.0.1:8501/_stcore/health"
ready = False
try:
    ready = requests.get(health_url, timeout=2).status_code == 200
except requests.RequestException:
    pass

if not ready:
    log_path = "/tmp/streamlit.log"
    log_file = open(log_path, "w")
    process = subprocess.Popen(
        [
            "streamlit", "run", "app.py",
            "--server.port", "8501",
            "--server.address", "0.0.0.0",
            "--server.headless", "true",
            "--server.enableCORS", "false",
            "--server.enableXsrfProtection", "false",
        ],
        cwd=str(project_dir),
        stdout=log_file,
        stderr=subprocess.STDOUT,
    )

    for _ in range(30):
        if process.poll() is not None:
            break
        try:
            response = requests.get(health_url, timeout=2)
            if response.status_code == 200:
                ready = True
                break
        except requests.RequestException:
            pass
        time.sleep(2)

if ready:
    print("Streamlit успешно запущен. Открываю интерфейс через прокси Colab...")
    output.serve_kernel_port_as_window(8501)
else:
    print("Не удалось запустить Streamlit. Журнал запуска:")
    print(Path("/tmp/streamlit.log").read_text(errors="replace"))
